# 04 — Gunakan kembali embedding untuk holdout 70:15:15
**Alur:** periksa Step03 baru → periksa embedding selesai → cocokkan sumber dan ID → buat paket baru.

Notebook ini tidak memuat PyTorch/Transformers, tidak mengunduh model, dan tidak menghitung ulang Bio-ClinicalBERT. Embedding beku tidak memerlukan fitting ulang ketika pasien dibagi ke subset yang berbeda.

File numerik, teks dan checkpoint lama hanya dibaca. Sumber embedding tetap menjadi dependensi; jangan hapus folder asalnya.


## 1. Konfigurasi
Simpan di `icu_los/notebooks`. Periksa tiga lokasi run di bawah, kemudian **Restart Kernel → Run All**. Dependensi: numpy, pandas, IPython.

Run sumber harus selesai dengan status `TEXT_EMBEDDINGS_COMPLETE_READY_FOR_MODEL_DESIGN`. Bila belum selesai atau file sumber berubah, proses berhenti; tidak otomatis menjalankan BERT.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from contextlib import contextmanager
import hashlib
import inspect
import json
import os
import time

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path(r"C:\Users\ASUS\Documents\icu_los")
STAGE03_RUN = "20261102_155934_731169_preprocess_70_15_15"
EMBEDDING_SOURCE_RUN = "20261020_080413_928636_text_embeddings_accelerated"
RESUME_RUN = ""  # Kosong: laporan baru. Isi ID run reuse ini untuk memverifikasi ulang.
MODEL_ID = "emilyalsentzer/Bio_ClinicalBERT"
MODEL_REVISION = "d5892b39a4adaed74b92212a44081509db72f87b"
STAGE03_READY = "SPLIT_PREPROCESS_COMPLETE_READY_FOR_TEXT_EMBEDDINGS"
READY = "TEXT_EMBEDDINGS_COMPLETE_READY_FOR_MODEL_DESIGN"
REPRESENTATIONS = [
    "icustay_ids_LOCAL.npy",
    "unique_document_vectors_LOCAL.npy",
    "unique_document_token_counts.npy",
    "unique_document_chunk_counts.npy",
    "diagnosis_vectors_LOCAL.npy",
    "diagnosis_available_hour_LOCAL.npy",
    "diagnosis_present_LOCAL.npy",
    "diagnosis_document_index_LOCAL.csv",
    "note_interval_vectors_LOCAL.npy",
    "note_interval_index_LOCAL.npy",
    "note_stay_offsets_LOCAL.npy",
    "note_document_index_LOCAL.csv",
]


## 2. Fungsi file dan integritas
Hash ekspor dicocokkan dengan receipt sumber. Pemeriksaan ini bisa memakan waktu karena membaca file besar, tetapi tidak melakukan inference. Isi target LOS tidak dimuat; hash berkas target hanya dapat ikut diverifikasi sebagai bagian integritas Step03.


In [2]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))


def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8"
    )
    os.replace(temporary, path)


def file_hash(path):
    digest = hashlib.sha256()
    updated = time.monotonic()
    size = 0
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
            size += len(chunk)
            if time.monotonic() - updated >= 25:
                print(f"Integrity: {Path(path).name}, {size / 2 ** 30:.2f} GiB", flush=True)
                updated = time.monotonic()
    return digest.hexdigest()


def artifact(path):
    path = Path(path).resolve()
    stat = path.stat()
    return dict(
        path=str(path),
        bytes=stat.st_size,
        mtime_ns=stat.st_mtime_ns,
        sha256=file_hash(path),
    )


def artifact_without_rehash(path, digest):
    path = Path(path).resolve()
    stat = path.stat()
    return dict(
        path=str(path), bytes=stat.st_size, mtime_ns=stat.st_mtime_ns, sha256=digest
    )


def verify_artifacts(entries, label):
    """Cocokkan file dengan hash yang sudah dicatat oleh run sumber."""
    verified = {}
    for i, entry in enumerate(entries):
        path = Path(entry["path"]).resolve()
        require(path.is_file(), f"{label}: file tidak ditemukan: {path}")
        require(
            path.stat().st_size == entry["bytes"] and file_hash(path) == entry["sha256"],
            f"{label}: isi file berubah: {path.name}",
        )
        verified[str(path)] = dict(entry, path=str(path), mtime_ns=path.stat().st_mtime_ns)
        if i % 25 == 0 or i == len(entries) - 1:
            print(f"{label}: {i + 1}/{len(entries)} files", flush=True)
    return verified


def require_pass_checks(path, allow_not_applicable=False):
    table = pd.read_csv(path)
    allowed = ["PASS", "NOT_APPLICABLE"] if allow_not_applicable else ["PASS"]
    require(
        len(table) > 0 and table.status.isin(allowed).all(),
        f"Quality checks belum lolos: {path}",
    )


def registered_artifacts(metadata, paths):
    registry = {str(Path(a["path"]).resolve()): a for a in metadata["artifacts"]}
    result = []
    for path in paths:
        key = str(Path(path).resolve())
        require(key in registry, f"File tidak tercatat pada receipt embedding: {key}")
        result.append(registry[key])
    return result


@contextmanager
def locked(path):
    f = open(path, "a+b")
    f.seek(0)
    f.write(b"0")
    f.flush()
    f.seek(0)
    try:
        if os.name == "nt":
            import msvcrt

            msvcrt.locking(f.fileno(), msvcrt.LK_NBLCK, 1)
        else:
            import fcntl

            fcntl.flock(f.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
    except (OSError, BlockingIOError):
        f.close()
        raise RuntimeError(
            "Run sedang aktif. Jangan jalankan dua kernel untuk run yang sama."
        )
    try:
        yield
    finally:
        f.seek(0)
        if os.name == "nt":
            msvcrt.locking(f.fileno(), msvcrt.LK_UNLCK, 1)
        else:
            fcntl.flock(f.fileno(), fcntl.LOCK_UN)
        f.close()


## 3. Periksa paket preprocessing holdout
Verifikasi keluaran Step03, row map, jadwal prediksi dan isolasi pasien. Tidak mengubah split atau melakukan fitting ulang.


In [3]:
def load_holdout(root, stage03_run):
    """Baca paket holdout selesai dan verifikasi seluruh file preprocessing."""
    report = Path(root) / "outputs" / stage03_run
    metadata_path = report / "run_metadata.json"
    metadata = read_json(metadata_path)
    require(metadata["status"] == STAGE03_READY, "Step03 belum selesai")
    package_path = Path(metadata["input_package"])
    require(
        file_hash(package_path) == metadata["input_package_sha256"], "Paket Step03 berubah"
    )
    package = read_json(package_path)
    require(
        package["version"] == "icu_los_v2_stage03_holdout_70_15_15"
        and package["fit_scope"] == "TRAIN_ONLY_HOLDOUT"
        and (package["split_protocol"] == "patient_random_holdout"),
        "Gunakan paket holdout 70:15:15",
    )
    require(
        package["split_ratios"] == {"train": 0.7, "validation": 0.15, "test": 0.15},
        "Rasio berbeda",
    )
    require(
        not package["target_feature"] and package["target_transform"] == "none",
        "Kontrak target berbeda",
    )
    require_pass_checks(report / "quality_checks.csv")
    watched = verify_artifacts(
        metadata["output_artifacts"] + metadata["sources"], "Step03 integrity"
    )
    watched[str(metadata_path.resolve())] = artifact(metadata_path)
    sample = pd.read_csv(package["row_map"])
    sequence = pd.read_csv(package["sequence_index"])
    assignments = pd.read_csv(package["split_assignments"])
    n = len(sample)
    require(
        n == metadata["counts"]["cohort_stays"]
        and sample.ICUSTAY_ID.is_unique
        and np.array_equal(sample.cohort_row, np.arange(n)),
        "Urutan cohort tidak valid",
    )
    require(
        np.array_equal(sequence.ICUSTAY_ID, sample.ICUSTAY_ID)
        and np.array_equal(sequence.cohort_row, sample.cohort_row),
        "Urutan sequence berbeda",
    )
    require(
        sequence.first_prediction_hour.eq(4).all()
        and sequence.last_prediction_hour.ge(4).all(),
        "Jadwal prediksi berbeda",
    )
    require(
        len(assignments) == n
        and np.array_equal(assignments.cohort_row, sample.cohort_row)
        and np.array_equal(assignments.ICUSTAY_ID, sample.ICUSTAY_ID)
        and np.array_equal(assignments.SUBJECT_ID, sample.SUBJECT_ID),
        "Pemetaan split berbeda",
    )
    require(
        assignments.groupby("SUBJECT_ID").split.nunique().eq(1).all()
        and set(assignments.split) == {"train", "validation", "test"},
        "Isolasi pasien gagal",
    )
    return dict(
        metadata=metadata,
        package=package,
        sample=sample,
        sequence=sequence,
        watched=watched,
    )


## 4. Periksa embedding yang sudah selesai
Gunakan ekspor dari run accelerated yang telah selesai. Pastikan identitas model, revision, dimensi 768, float32 serta bukti sumber tokenisasi/pooling cocok. Audit token dan agregasi dari run lama diwarisi melalui receipt; tidak dihitung ulang di sini.


In [4]:
def load_completed_embeddings(root, embedding_run):
    """Verifikasi ekspor embedding selesai; tidak membuka encoder BERT."""
    report = Path(root) / "outputs" / embedding_run
    metadata_path = report / "run_metadata.json"
    metadata = read_json(metadata_path)
    require(
        metadata["status"] == READY,
        "Embedding sumber belum selesai; jangan jalankan relink",
    )
    package_path = Path(metadata["text_input_package"])
    require(
        file_hash(package_path) == metadata["text_input_package_sha256"],
        "Paket embedding berubah",
    )
    package = read_json(package_path)
    require(
        package["version"] == "icu_los_v2_stage04", "Versi embedding sumber tidak sesuai"
    )
    require(
        not package["train_fitted_transform"] and (not package["LOS_target_in_X"]),
        "Embedding sumber bergantung target/fit train",
    )
    encoder = package["encoder"]
    require(
        encoder["model_id"] == MODEL_ID
        and encoder["revision"] == MODEL_REVISION
        and (encoder["hidden_size"] == 768)
        and (encoder["dtype"] == "float32")
        and (encoder["attention"] == "eager"),
        "Identitas encoder berbeda",
    )
    require(
        all(
            (
                encoder[k] == metadata["encoder"][k]
                for k in [
                    "model_id",
                    "revision",
                    "hidden_size",
                    "dtype",
                    "attention",
                    "tokenizer_sha256",
                ]
            )
        ),
        "Encoder pada metadata dan paket berbeda",
    )
    paths = package["representations"]
    require(set(REPRESENTATIONS).issubset(paths), "Ekspor representasi belum lengkap")
    required_paths = [paths[name] for name in REPRESENTATIONS]
    required_paths += [
        package_path,
        package["catalog_LOCAL"],
        package["checkpoints_ledger"],
        report / "quality_checks.csv",
    ]
    watched = verify_artifacts(
        registered_artifacts(metadata, required_paths), "Embedding integrity"
    )
    require_pass_checks(report / "quality_checks.csv", allow_not_applicable=True)
    watched[str(metadata_path.resolve())] = artifact(metadata_path)
    return dict(metadata=metadata, package=package, watched=watched, report=report)


## 5. Cocokkan sumber lama dan baru
Perubahan yang diizinkan hanya split dan preprocessing numerik. Cohort, urutan stay, sumber Step02, database teks, diagnosis dan waktu ketersediaannya harus sama.

Notebook ini memerlukan metadata Step03 lama, metadata run embedding asal, serta receipt migrasi accelerated untuk membuktikan asal-usul hasil. Biarkan folder lama tetap tersedia.


In [5]:
def verify_shared_sources(root, holdout, embeddings):
    """Split boleh berubah; cohort, row map, urutan waktu dan sumber teks harus sama."""
    new = holdout["package"]
    old = embeddings["package"]
    require(
        new["cohort_sha256"]
        == old["cohort_sha256"]
        == embeddings["metadata"]["cohort_sha256"],
        "Cohort berbeda: embedding tidak boleh digunakan kembali",
    )
    require(len(holdout["sample"]) == old["cohort_stays"], "Jumlah stay embedding berbeda")
    old_package_path = Path(old["stage03_input_package"])
    require(
        file_hash(old_package_path) == old["stage03_input_package_sha256"],
        "Paket preprocessing lama berubah",
    )
    previous = read_json(old_package_path)
    old_metadata_path = old_package_path.parent / "run_metadata.json"
    previous_metadata = read_json(old_metadata_path)
    require(
        previous_metadata["status"] == STAGE03_READY
        and previous_metadata["input_package_sha256"]
        == old["stage03_input_package_sha256"],
        "Receipt preprocessing lama berbeda",
    )
    require(
        previous["cohort_sha256"] == new["cohort_sha256"],
        "Cohort preprocessing lama berbeda",
    )
    require(
        previous_metadata["stage02_run"] == holdout["metadata"]["stage02_run"],
        "Sumber Step02 berbeda",
    )
    require(
        previous["prediction"] == new["prediction"]
        or (
            previous["prediction"]["first_hour"] == new["prediction"]["first_hour"] == 4
            and previous["prediction"]["step_hours"] == new["prediction"]["step_hours"] == 1
        ),
        "Jadwal waktu berbeda",
    )
    for key in [
        "row_map",
        "row_map_sha256",
        "sequence_index",
        "notes_database",
        "notes_table",
        "diagnosis_file",
    ]:
        require(previous[key] == new[key], "Sumber berbeda: " + key)
    original_run = embeddings["metadata"]["source_run"]
    original_path = Path(root) / "outputs" / original_run / "run_metadata.json"
    original = read_json(original_path)
    require(
        original["settings"]["overlap"] == 64
        and original["settings"]["content_capacity"] == 510
        and (
            original["settings"]["pooling"] == "content_mean_and_inverse_overlap_coverage"
        ),
        "Metode tokenisasi/pooling sumber berbeda",
    )
    for key in [
        "model_id",
        "revision",
        "hidden_size",
        "dtype",
        "attention",
        "tokenizer_sha256",
    ]:
        require(
            original["encoder"][key] == old["encoder"][key], "Encoder asal berbeda: " + key
        )
    migration_path = embeddings["report"] / "migration_commit.json"
    verification_path = (
        Path(embeddings["metadata"]["data_directory"]) / "source_database_verification.json"
    )
    extra = registered_artifacts(
        embeddings["metadata"], [migration_path, verification_path]
    )
    watched = verify_artifacts(extra, "Source receipt integrity")
    migration = read_json(migration_path)
    require(
        migration["source_metadata_sha256"] == file_hash(original_path),
        "Metadata embedding asal berubah",
    )
    watched.update(
        verify_artifacts(original["settings"]["small_sources"], "Original source integrity")
    )
    identity = read_json(verification_path)
    stage02 = read_json(
        Path(root) / "outputs" / holdout["metadata"]["stage02_run"] / "run_metadata.json"
    )
    require(
        identity["sha256"] == stage02["database_sha256"], "Identitas database teks berbeda"
    )
    database = Path(new["notes_database"])
    wal = Path(str(database) + "-wal")
    require(not wal.exists() or wal.stat().st_size == 0, "Database teks memiliki WAL aktif")
    require(
        database.stat().st_size == identity["bytes"]
        and file_hash(database) == identity["sha256"],
        "Database teks sumber berubah",
    )
    watched[str(database.resolve())] = artifact_without_rehash(database, identity["sha256"])
    for path in [old_package_path, old_metadata_path, original_path]:
        watched[str(path.resolve())] = artifact(path)
    return watched


## 6. Periksa bentuk dan waktu representasi
Periksa seluruh urutan ICU stay, shape vektor, offset notes, nilai finite dan batas waktu. Tidak mengubah isi array. Diagnosis kosong memakai zero + mask; notes kosong tetap tidak di-forward-fill.


In [6]:
def verify_representation_arrays(package, sample, sequence):
    """Periksa seluruh ID, dimensi, nilai, offset, dan waktu representasi."""
    paths = package["representations"]
    arrays = {
        name: np.load(paths[name], mmap_mode="r", allow_pickle=False)
        for name in REPRESENTATIONS
        if name.endswith(".npy")
    }
    n = len(sample)
    require(
        np.array_equal(arrays["icustay_ids_LOCAL.npy"], sample.ICUSTAY_ID.to_numpy()),
        "Urutan embedding/stay berbeda",
    )
    documents = arrays["unique_document_vectors_LOCAL.npy"]
    require(documents.ndim == 2 and documents.shape[1] == 768, "Dimensi dokumen bukan 768")
    for name in ["unique_document_token_counts.npy", "unique_document_chunk_counts.npy"]:
        a = arrays[name]
        require(
            a.shape == (len(documents),)
            and np.issubdtype(a.dtype, np.integer)
            and (a > 0).all(),
            "Jumlah token/chunk invalid",
        )
    diagnosis = arrays["diagnosis_vectors_LOCAL.npy"]
    hours = arrays["diagnosis_available_hour_LOCAL.npy"]
    present = arrays["diagnosis_present_LOCAL.npy"]
    require(
        diagnosis.shape == (n, 768) and hours.shape == present.shape == (n,),
        "Shape diagnosis berbeda",
    )
    require(np.isin(present, [0, 1]).all(), "Mask diagnosis invalid")
    selected = present.astype(bool)
    require(
        np.isfinite(hours[selected]).all()
        and (hours[selected] < sequence.last_prediction_hour.to_numpy()[selected]).all(),
        "Waktu diagnosis invalid",
    )
    require(
        np.isposinf(hours[~selected]).all() and (diagnosis[~selected] == 0).all(),
        "Diagnosis kosong bukan zero/mask",
    )
    index = arrays["note_interval_index_LOCAL.npy"]
    offsets = arrays["note_stay_offsets_LOCAL.npy"]
    notes = arrays["note_interval_vectors_LOCAL.npy"]
    require(
        index.ndim == 2 and index.shape[1] == 3 and (notes.shape == (len(index), 768)),
        "Shape notes invalid",
    )
    require(
        np.issubdtype(index.dtype, np.integer) and np.issubdtype(offsets.dtype, np.integer),
        "Indeks notes harus integer",
    )
    require(
        offsets.shape == (n + 1,)
        and offsets[0] == 0
        and (offsets[-1] == len(index))
        and (np.diff(offsets) >= 0).all(),
        "Offset notes invalid",
    )
    if len(index):
        row, hour, count = index.T
        require(
            ((row >= 0) & (row < n) & (hour >= 0) & (count > 0)).all(),
            "Indeks notes di luar batas",
        )
        require(
            (hour < sequence.last_prediction_hour.to_numpy()[row]).all(),
            "Interval notes melewati akhir stay",
        )
        require(
            np.array_equal(row, np.repeat(np.arange(n), np.diff(offsets))),
            "Offset tidak cocok dengan row notes",
        )
        ordered = (row[1:] > row[:-1]) | (row[1:] == row[:-1]) & (hour[1:] > hour[:-1])
        require(ordered.all(), "Interval notes tidak unik/terurut")
    for name in [
        "unique_document_vectors_LOCAL.npy",
        "diagnosis_vectors_LOCAL.npy",
        "note_interval_vectors_LOCAL.npy",
    ]:
        values = arrays[name]
        require(values.dtype == np.float32, "Embedding harus float32: " + name)
        for start in range(0, len(values), 8192):
            require(
                np.isfinite(values[start : start + 8192]).all(),
                "Embedding tidak finite: " + name,
            )
    counts = dict(
        cohort_stays=n,
        unique_texts=len(documents),
        diagnosis_documents=int(present.sum()),
        note_nonempty_intervals=len(index),
        note_documents=int(index[:, 2].sum()),
    )
    for values in arrays.values():
        if hasattr(values, "_mmap") and values._mmap is not None:
            values._mmap.close()
    return counts


## 7. Loader prefix dan pemeriksaan sampel
Persamaan loader dipertahankan dari Step04 sebelumnya. Notes menggunakan interval baru dengan bin `h < t`. Diagnosis masuk sesuai gate waktu ketersediaannya. Pemeriksaan prefix di sini bukan evaluasi akurasi model LOS.


In [7]:
def load_text_prefix(text_package, cohort_row, prediction_hour):
    """Reference loader for a single stay prefix. No target, outcome or final length returned.

    Caller verifies that prediction_hour is an eligible schedule point externally.
    Notes are NEW documents per interval, not a cumulative text embedding.
    """
    require(
        isinstance(prediction_hour, (int, np.integer)) and prediction_hour >= 4,
        "Gunakan jam prediksi bulat >=4",
    )
    p = read_json(text_package) if isinstance(text_package, (str, Path)) else text_package
    paths = p["representations"]
    row = int(cohort_row)
    t = int(prediction_hour)
    n = p["cohort_stays"]
    require(0 <= row < n, "Row invalid")
    offsets = np.load(paths["note_stay_offsets_LOCAL.npy"], mmap_mode="r")
    index = np.load(paths["note_interval_index_LOCAL.npy"], mmap_mode="r")
    vectors = np.load(paths["note_interval_vectors_LOCAL.npy"], mmap_mode="r")
    lo, hi = offsets[row : row + 2]
    group = index[lo:hi]
    keep = group[:, 1] < t
    group = group[keep]
    selected = vectors[lo:hi][keep]
    notes = np.zeros((t, 768), dtype=np.float32)
    mask = np.zeros(t, dtype=np.uint8)
    notes[group[:, 1]] = selected
    mask[group[:, 1]] = 1
    present = np.load(paths["diagnosis_present_LOCAL.npy"], mmap_mode="r")
    hours = np.load(paths["diagnosis_available_hour_LOCAL.npy"], mmap_mode="r")
    gate = (np.arange(1, t + 1, dtype=np.float64) > hours[row]) & bool(present[row])
    raw = np.load(paths["diagnosis_vectors_LOCAL.npy"], mmap_mode="r")[row]
    diagnosis = (gate[:, None] * raw[None, :]).astype(np.float32)
    return dict(
        notes=notes,
        note_mask=mask,
        diagnosis=diagnosis,
        diagnosis_mask=gate.astype(np.uint8),
    )


def check_prefix_loader(package):
    """Periksa cutoff jam dan konsistensi prefix pada sampel representasi."""
    paths = package["representations"]
    present = np.load(paths["diagnosis_present_LOCAL.npy"], allow_pickle=False)
    hours = np.load(paths["diagnosis_available_hour_LOCAL.npy"], allow_pickle=False)
    rows = {0, package["cohort_stays"] - 1}
    rows.update(np.flatnonzero((present == 1) & (hours >= 4))[:3].tolist())
    index = np.load(
        paths["note_interval_index_LOCAL.npy"], mmap_mode="r", allow_pickle=False
    )
    rows.update(index[:3, 0].tolist())
    for row in rows:
        short = load_text_prefix(package, row, 4)
        long = load_text_prefix(package, row, 6)
        require(
            all((np.array_equal(short[k], long[k][:4]) for k in short)),
            "Prefix berubah oleh masa depan",
        )
        require(
            (short["notes"][short["note_mask"] == 0] == 0).all(), "Notes kosong bukan zero"
        )
        expected = (np.arange(1, 7) > hours[row]) & bool(present[row])
        require(
            np.array_equal(long["diagnosis_mask"], expected), "Time gate diagnosis berbeda"
        )
    index._mmap.close()
    return len(rows)


## 8. Buat kontrak input baru
Paket baru menunjuk hasil preprocessing holdout, tetapi tetap merujuk file representasi lama yang sama. Tidak menyalin array besar atau mengganti fingerprint sumber.


In [8]:
def build_relinked_package(holdout, embeddings, source_run):
    """Buat kontrak baru; file representasi lama hanya dirujuk, tidak disalin."""
    previous = embeddings["package"]
    package = dict(previous)
    package.pop("shared_across_folds", None)
    package.update(
        version="icu_los_v2_stage04_holdout_70_15_15",
        stage03_input_package=holdout["metadata"]["input_package"],
        stage03_input_package_sha256=holdout["metadata"]["input_package_sha256"],
        split_protocol="patient_random_holdout",
        split_ratios=holdout["package"]["split_ratios"],
        embedding_source_run=source_run,
        embedding_source_package_sha256=embeddings["metadata"]["text_input_package_sha256"],
        shared_across_splits=True,
        bert_recomputed=False,
        reuse_scope="verified existing cohort-aligned representations; no numerical changes",
    )
    return package


def check_files_unchanged(watched):
    for entry in watched.values():
        stat = Path(entry["path"]).stat()
        require(
            stat.st_size == entry["bytes"] and stat.st_mtime_ns == entry["mtime_ns"],
            "Sumber berubah selama proses: " + Path(entry["path"]).name,
        )


## 9. Simpan laporan
Run reuse memiliki metadata dan receipt sendiri. Hash sumber dicatat terpisah dari hash output baru. Jika file berubah saat diperiksa, hasil tidak ditandai selesai.


In [9]:
def code_hash():
    import types

    def value(x):
        if isinstance(x, types.CodeType):
            return dict(
                bytecode=x.co_code.hex(),
                constants=[value(v) for v in x.co_consts],
                names=x.co_names,
                varnames=x.co_varnames,
                freevars=x.co_freevars,
                cellvars=x.co_cellvars,
                args=x.co_argcount,
                posonly=x.co_posonlyargcount,
                kwonly=x.co_kwonlyargcount,
                flags=x.co_flags,
                exceptions=getattr(x, "co_exceptiontable", b"").hex(),
            )
        if isinstance(x, tuple):
            return {"tuple": [value(v) for v in x]}
        if isinstance(x, frozenset):
            return {"frozenset": sorted([value(v) for v in x], key=repr)}
        if isinstance(x, bytes):
            return {"bytes": x.hex()}
        if x is None or isinstance(x, (str, int, float, bool)):
            return x
        return {"type": type(x).__name__, "repr": repr(x)}

    payload = {k: value(inspect.unwrap(globals()[k]).__code__) for k in ENGINE_FUNCTIONS}
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, ensure_ascii=True).encode()
    ).hexdigest()


def save_relink_report(root, holdout, embeddings, package, watched, counts, prefix_samples):
    run_id = (
        RESUME_RUN
        or datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
        + "_text_embeddings_70_15_15_reuse"
    )
    require(
        Path(run_id).name == run_id and "/" not in run_id and ("\\" not in run_id),
        "Nama run tidak valid",
    )
    require(
        run_id not in [STAGE03_RUN, EMBEDDING_SOURCE_RUN],
        "Run tujuan harus berbeda dari sumber",
    )
    report = Path(root) / "outputs" / run_id
    data = Path(root) / "data" / run_id
    report.mkdir(parents=True, exist_ok=True)
    data.mkdir(parents=True, exist_ok=True)
    fingerprint = hashlib.sha256(
        json.dumps(
            {
                "code": code_hash(),
                "new_stage03": holdout["metadata"]["input_package_sha256"],
                "old_stage04": embeddings["metadata"]["text_input_package_sha256"],
                "sources": {p: e["sha256"] for p, e in watched.items()},
            },
            sort_keys=True,
        ).encode()
    ).hexdigest()
    with locked(data / "active_run.lock"):
        metadata_path = report / "run_metadata.json"
        if metadata_path.exists():
            old = read_json(metadata_path)
            require(
                RESUME_RUN and old["fingerprint"] == fingerprint,
                "Run sudah ada atau sumber/kode berubah",
            )
            for entry in old["artifacts"]:
                verify_artifacts([entry], "Saved output")
            print("Run relink sudah selesai dan terverifikasi.")
            return old
        atomic_json(report / "text_input_package.json", package)
        checks = [
            "holdout_preprocessing_artifacts_verified",
            "completed_embedding_package_and_exports_verified",
            "same_cohort_row_order_stage02_and_text_sources",
            "same_frozen_encoder_revision_and_pooling",
            "all_representation_shapes_offsets_finite_values_and_availability",
            "sample_prefix_loader_causal_and_diagnosis_gate",
            "no_BERT_recomputation_or_fitting_or_target_loading",
            "source_sizes_mtimes_unchanged",
        ]
        pd.DataFrame({"check": checks, "status": "PASS"}).to_csv(
            report / "quality_checks.csv", index=False
        )
        pd.DataFrame([{"metric": k, "value": v} for k, v in counts.items()]).to_csv(
            report / "embedding_counts.csv", index=False
        )
        atomic_json(
            report / "reuse_receipt.json",
            dict(
                source_run=EMBEDDING_SOURCE_RUN,
                source_package_sha256=embeddings["metadata"]["text_input_package_sha256"],
                new_stage03_run=STAGE03_RUN,
                new_stage03_package_sha256=holdout["metadata"]["input_package_sha256"],
                source_artifacts=list(watched.values()),
                prefix_sample_stays=prefix_samples,
                re_inference_performed=False,
                aggregation_recomputed=False,
                test_scored=False,
            ),
        )
        outputs = [
            artifact(report / name)
            for name in [
                "text_input_package.json",
                "quality_checks.csv",
                "embedding_counts.csv",
                "reuse_receipt.json",
            ]
        ]
        metadata = dict(
            run_id=run_id,
            stage="04_text_embeddings_holdout_reuse",
            status=READY,
            fingerprint=fingerprint,
            code_sha256=code_hash(),
            stage03_run=STAGE03_RUN,
            source_run=EMBEDDING_SOURCE_RUN,
            cohort_sha256=package["cohort_sha256"],
            data_directory=str(data.resolve()),
            report_directory=str(report.resolve()),
            encoder=package["encoder"],
            counts=counts,
            artifacts=outputs,
            text_input_package=str((report / "text_input_package.json").resolve()),
            text_input_package_sha256=file_hash(report / "text_input_package.json"),
            quality_checks=len(checks),
            bert_recomputed=False,
            test_scored=False,
        )
        check_files_unchanged(watched)
        atomic_json(metadata_path, metadata)
        summary = "\n".join(
            [
                "ICU LOS v2 — STEP04 HOLDOUT EMBEDDING REUSE",
                "Run: " + run_id,
                "Step03: " + STAGE03_RUN,
                "Embedding source: " + EMBEDDING_SOURCE_RUN,
                f"Cohort: {counts['cohort_stays']:,} stays; embedding dimension: 768",
                "Split: 70:15:15 at patient level.",
                "Original embedding arrays referenced unchanged; retain their source folders.",
                "Original sparse note intervals and diagnosis availability gates preserved.",
                "No BERT loading, tokenization, inference, fitted transform or test scoring.",
                "Export hashes and metadata verified; original semantic audits inherited, not rerun.",
                f"Checks: {len(checks)} PASS",
                "Status: " + READY,
                "Reports: " + str(report),
            ]
        )
        (report / "summary.txt").write_text(summary, encoding="utf-8")
        print(summary)
        return metadata


In [10]:
ENGINE_FUNCTIONS = [
    "require",
    "read_json",
    "atomic_json",
    "file_hash",
    "artifact",
    "verify_artifacts",
    "require_pass_checks",
    "registered_artifacts",
    "load_holdout",
    "load_completed_embeddings",
    "verify_shared_sources",
    "artifact_without_rehash",
    "verify_representation_arrays",
    "check_prefix_loader",
    "build_relinked_package",
    "check_files_unchanged",
    "save_relink_report",
    "run_embedding_reuse",
    "locked",
    "load_text_prefix",
    "code_hash",
]


## 10. Jalankan
Semua pemeriksaan dilakukan sebelum paket dinyatakan siap. Setiap model pada Step05 nanti menggunakan split pasien dan preprocessing dari Step03 baru, bukan fold lama.


In [11]:
def run_embedding_reuse(root, stage03_run, embedding_run):
    holdout = load_holdout(root, stage03_run)
    embeddings = load_completed_embeddings(root, embedding_run)
    extra_sources = verify_shared_sources(root, holdout, embeddings)
    counts = verify_representation_arrays(
        embeddings["package"], holdout["sample"], holdout["sequence"]
    )
    for name, count in counts.items():
        require(
            count == embeddings["metadata"]["counts"][name],
            "Jumlah ekspor berbeda: " + name,
        )
    prefix_samples = check_prefix_loader(embeddings["package"])
    package = build_relinked_package(holdout, embeddings, embedding_run)
    watched = {**holdout["watched"], **embeddings["watched"], **extra_sources}
    check_files_unchanged(watched)
    return save_relink_report(
        root, holdout, embeddings, package, watched, counts, prefix_samples
    )


In [12]:
metadata = run_embedding_reuse(ROOT, STAGE03_RUN, EMBEDDING_SOURCE_RUN)
report_dir = Path(metadata["report_directory"])
display(pd.read_csv(report_dir / "quality_checks.csv"))
display(pd.read_csv(report_dir / "embedding_counts.csv"))
print("STAGE04_RUN =", repr(metadata["run_id"]))


Step03 integrity: 1/1264 files
Step03 integrity: 26/1264 files
Step03 integrity: 51/1264 files
Step03 integrity: 76/1264 files
Step03 integrity: 101/1264 files
Step03 integrity: 126/1264 files
Step03 integrity: 151/1264 files
Step03 integrity: 176/1264 files
Step03 integrity: 201/1264 files
Step03 integrity: 226/1264 files
Step03 integrity: 251/1264 files
Step03 integrity: 276/1264 files
Step03 integrity: 301/1264 files
Step03 integrity: 326/1264 files
Step03 integrity: 351/1264 files
Step03 integrity: 376/1264 files
Step03 integrity: 401/1264 files
Step03 integrity: 426/1264 files
Step03 integrity: 451/1264 files
Step03 integrity: 476/1264 files
Step03 integrity: 501/1264 files
Step03 integrity: 526/1264 files
Step03 integrity: 551/1264 files
Step03 integrity: 576/1264 files
Step03 integrity: 601/1264 files
Step03 integrity: 626/1264 files
Step03 integrity: 651/1264 files
Step03 integrity: 676/1264 files
Step03 integrity: 701/1264 files
Step03 integrity: 726/1264 files
Step03 integrit

,check,status
0,holdout_preprocessing_artifacts_verified,PASS
1,completed_embedding_package_and_exports_verified,PASS
2,same_cohort_row_order_stage02_and_text_sources,PASS
3,same_frozen_encoder_revision_and_pooling,PASS
4,all_representation_shapes_offsets_finite_value...,PASS
5,sample_prefix_loader_causal_and_diagnosis_gate,PASS
6,no_BERT_recomputation_or_fitting_or_target_loa...,PASS
7,source_sizes_mtimes_unchanged,PASS


,metric,value
0,cohort_stays,53044
1,unique_texts,1013646
2,diagnosis_documents,53039
3,note_nonempty_intervals,800532
4,note_documents,1033980


STAGE04_RUN = '20261102_161545_111298_text_embeddings_70_15_15_reuse'


## 11. Ringkasan untuk Step05
Gunakan `STAGE03_RUN` dan ID `STAGE04_RUN` yang tercetak. File `text_input_package.json` memiliki versi holdout baru dan menautkan hasil Step03 70:15:15. Step05 lama masih perlu disesuaikan sebelum training.

Kirim `summary.txt`, `quality_checks.csv`, dan `embedding_counts.csv` setelah selesai. Tidak perlu mengirim teks klinis atau ID pasien.

Catatan: hash ekspor memastikan hasil lama tidak berubah. Notebook ini tidak mengulang seluruh audit semantik dokumen dan tidak melakukan re-inference BERT. Aturan timestamp proxy dan batasan teks dari run lama tetap berlaku.
